# Модуль 2. Разбор кейсов: Проверяемый цифровой верстак

**Модуль 2. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** molecules.csv; системные версии и контрольные значения концентрации.

Проверить интерпретатор, зависимость и контракт численного преобразования до химического конвейера.

**Workflow:** Интерпретатор → версии → hash входа → функция с единицами → отрицательные контроли.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab02_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%202_Setup%20Workharness.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [1]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab02/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


Python: /home/miakhlov/course-notebooks/.venv/bin/python


## 1. Редактор, ядро и интерпретатор

Ноутбук — документ; вычисления выполняет отдельный Python-процесс. Поэтому записываем sys.executable, версии и SHA256 входа до расчёта. Файл одинакового имени может содержать другие байты. Hash позволяет идентифицировать snapshot, но ничего не сообщает о правильности химической записи.

**Перед запуском:** Запишите путь Python и версии, отличая среду выполнения от редактора.

In [2]:
from chemistry import p_activity
import hashlib, platform, rdkit
passport = {"python": platform.python_version(), "executable": sys.executable,
            "rdkit": rdkit.__version__, "sha256": hashlib.sha256((DATA/"molecules.csv").read_bytes()).hexdigest()}
print(passport)


{'python': '3.12.3', 'executable': '/home/miakhlov/course-notebooks/.venv/bin/python', 'rdkit': '2026.09.1', 'sha256': 'f1904ebd8af5789d383f9593365bf5443abf7aa60106db0ecfb50d4fff628465'}


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Редактор, ядро и интерпретатор — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Проверка единиц на эквивалентных входах

Для pIC50 сначала переводим положительную концентрацию в моль/л, затем берём −log10. Значения 100 nM и 0,1 uM описывают одну концентрацию и обязаны совпасть после преобразования. Сравниваем float с допуском, а не строковое представление числа.

**Перед запуском:** Посчитайте SHA256 входа и pIC50 двумя эквивалентными способами.

In [3]:
assert math.isclose(p_activity(100,"nM"),7)
assert math.isclose(p_activity(0.1,"uM"),7)


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Проверка единиц на эквивалентных входах — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Контролируемый отказ

Ноль не имеет конечного отрицательного логарифма; неизвестная единица не допускает однозначного пересчёта. Программа должна вернуть понятную ошибку, а не заменять значение нулём. Отрицательный контроль проверяет контракт функции. В IDE выбираем ту же .venv, что указана в паспорте ядра.

**Перед запуском:** Проверьте отрицательный/нулевой ввод и неизвестную единицу без молчаливого исправления.

In [4]:
for value,unit in [(0,"nM"),(-1,"M"),(1,"unknown")]:
    try:
        p_activity(value,unit)
    except ValueError:
        continue
    raise AssertionError("Некорректный ввод принят")
artifact = passport


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Контролируемый отказ — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

100 nM = 0.1 uM → 7; ноль/неизвестная единица отклоняются; версии и hash сохранены.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [5]:
record={'module':2,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

/home/miakhlov/course-notebooks/Labs/results/lab02/worked/benchmark.json


## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab02_practice.ipynb](lab02_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.